# Análise Exploratória

Visualizações interativas (Plotly) dos dados Gold.

**Pré-requisito:** `03_gold.ipynb` já executado (gold populado).

| Seção | Conteúdo |
|---|---|
| 5.1 | Evolução do gap salarial por UF e trimestre (PNAD) |
| 5.2 | Saldo de empregos por sexo e UF (CAGED) |
| 5.3 | Gap salarial por nível de escolaridade (RAIS) |
| 5.4 | Participação feminina por setor (RAIS) |
| 5.5 | Gap salarial por setor (RAIS) |

In [4]:
import sys, pathlib

_root = pathlib.Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [5]:
from config import config
import pandas as pd
import plotly.express as px

## 5.1 — Evolução do Gap Salarial por UF (PNAD)

In [6]:
df_gap = pd.read_csv(config.gold_dir / "gap_salarial_por_uf_sexo_idade.csv", encoding="utf-8")

_UF_NOMES = {41: "Paraná", 42: "Santa Catarina", 43: "Rio Grande do Sul"}
df_gap["uf"] = df_gap["uf_cod"].apply(lambda x: _UF_NOMES.get(int(x), str(x)))

def _fmt_periodo(p: int) -> str:
    s = str(int(p))
    return f"T{int(s[4:])} {s[:4]}"

df_gap["periodo_label"] = df_gap["periodo"].apply(_fmt_periodo)

df_gap_uf = (
    df_gap.groupby(["uf", "periodo", "periodo_label"])["gap_pct"]
    .mean()
    .reset_index()
    .sort_values("periodo")
)

fig = px.line(
    df_gap_uf,
    x="periodo_label",
    y="gap_pct",
    color="uf",
    title="Evolução do Gap Salarial de Gênero — Região Sul (%)",
    labels={"gap_pct": "Gap (%)", "periodo_label": "Trimestre", "uf": "Estado"},
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Paridade salarial")
fig.update_layout(
    xaxis_tickangle=45,
    xaxis_title="Trimestre",
    yaxis_title="Gap salarial (%)",
    legend_title_text="Estado",
    hovermode="x unified",
)
fig.show()

## 5.2 — Saldo de Empregos por Sexo e UF (CAGED)

In [7]:
df_vagas = pd.read_csv(config.gold_dir / "vagas_saldo_por_uf_perfil.csv", encoding="utf-8")
df_vagas = df_vagas[df_vagas["sexo"].isin(["Masculino", "Feminino"])]

df_saldo_sexo = (
    df_vagas.groupby(["ano", "sigla_uf", "sexo"])["saldo"]
    .sum()
    .reset_index()
)
df_saldo_sexo["ano"] = df_saldo_sexo["ano"].astype(str)

fig2 = px.bar(
    df_saldo_sexo,
    x="ano",
    y="saldo",
    color="sexo",
    color_discrete_map={"Masculino": "#4C72B0", "Feminino": "#DD8452"},
    barmode="group",
    facet_col="sigla_uf",
    facet_col_spacing=0.08,
    title="Saldo Líquido de Empregos por Sexo e UF (CAGED)",
    labels={"saldo": "Saldo líquido", "ano": "Ano", "sexo": "Sexo"},
)
fig2.update_layout(legend_title_text="Sexo", yaxis_title="Saldo líquido de empregos")
fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig2.show()

## 5.3 — Gap Salarial por Nível de Escolaridade (RAIS)

In [8]:
from analise.gap_salarial import ESCOLARIDADE_ORDEM

df_esc = pd.read_csv(config.gold_dir / "gap_salarial_por_escolaridade.csv", encoding="utf-8")

df_esc_agg = (
    df_esc.groupby("escolaridade")[["salario_masculino", "salario_feminino", "gap_pct"]]
    .mean()
    .round(2)
    .reset_index()
)
df_esc_agg["escolaridade"] = pd.Categorical(
    df_esc_agg["escolaridade"], categories=ESCOLARIDADE_ORDEM, ordered=True
)
df_esc_agg = df_esc_agg.sort_values("escolaridade")

print("Gap salarial por nível de escolaridade — Região Sul (RAIS)")
display(df_esc_agg.rename(columns={
    "escolaridade": "Escolaridade",
    "salario_masculino": "Salário médio M (R$)",
    "salario_feminino": "Salário médio F (R$)",
    "gap_pct": "Gap (%)",
}))

fig3 = px.bar(
    df_esc_agg,
    x="escolaridade",
    y="gap_pct",
    title="Gap Salarial de Gênero por Nível de Escolaridade — Região Sul (RAIS)",
    labels={"gap_pct": "Gap (%)", "escolaridade": "Escolaridade"},
    color="gap_pct",
    color_continuous_scale="RdYlGn_r",
    text="gap_pct",
)
fig3.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig3.update_layout(
    xaxis_title="Nível de escolaridade",
    yaxis_title="Gap salarial (%)",
    coloraxis_showscale=False,
    xaxis={"categoryorder": "array", "categoryarray": ESCOLARIDADE_ORDEM},
)
fig3.show()

Gap salarial por nível de escolaridade — Região Sul (RAIS)


,Escolaridade,Salário médio M (R$),Salário médio F (R$),Gap (%)
0,Analfabeto,2295.65,1745.43,31.45
2,Fund. incompleto,2385.41,1626.31,46.25
1,Fund. completo,2541.86,1753.16,45.01
4,Médio incompleto,2260.84,1656.96,36.36
3,Médio completo,2746.85,2098.58,30.83
7,Superior incompleto,3691.12,2550.58,44.25
6,Superior completo,7913.54,5043.73,56.29
5,Pós-graduação,11651.63,8721.37,33.54


## 5.4 — Participação Feminina por Setor (RAIS)

In [9]:
df_setor = pd.read_csv(config.gold_dir / "gap_salarial_por_setor.csv", encoding="utf-8")

df_fem = df_setor[df_setor["sexo"] == "Feminino"].copy()
df_setor_agg = (
    df_fem.groupby("setor")[["share_pct", "gap_pct", "salario_medio"]]
    .mean()
    .round(2)
    .sort_values("share_pct")
    .reset_index()
)

fig4 = px.bar(
    df_setor_agg,
    x="share_pct",
    y="setor",
    orientation="h",
    title="Participação feminina por setor de atividade (%) — Região Sul (RAIS)",
    labels={"share_pct": "Participação feminina (%)", "setor": "Setor"},
    color="share_pct",
    color_continuous_scale="RdYlGn",
    text="share_pct",
)
fig4.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig4.add_vline(x=50, line_dash="dash", line_color="gray", annotation_text="50% — paridade")
fig4.update_layout(
    xaxis_title="Participação feminina (%)",
    coloraxis_showscale=False,
    xaxis_range=[0, 100],
)
fig4.show()

## 5.5 — Gap Salarial por Setor (RAIS)

In [10]:
fig5 = px.bar(
    df_setor_agg.dropna(subset=["gap_pct"]).sort_values("gap_pct", ascending=False),
    x="gap_pct",
    y="setor",
    orientation="h",
    title="Gap Salarial de Gênero por Setor (%) — Região Sul (RAIS)",
    labels={"gap_pct": "Gap salarial (%)", "setor": "Setor"},
    color="gap_pct",
    color_continuous_scale="RdYlGn_r",
    text="gap_pct",
)
fig5.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig5.add_vline(x=0, line_dash="dash", line_color="gray")
fig5.update_layout(xaxis_title="Gap salarial (%)", coloraxis_showscale=False)
fig5.show()

print("\nResumo: setores com maior gap salarial e menor participação feminina")
display(
    df_setor_agg[["setor", "share_pct", "gap_pct", "salario_medio"]]
    .rename(columns={
        "setor": "Setor",
        "share_pct": "Part. feminina (%)",
        "gap_pct": "Gap salarial (%)",
        "salario_medio": "Salário médio F (R$)",
    })
    .sort_values("Gap salarial (%)", ascending=False)
)


Resumo: setores com maior gap salarial e menor participação feminina


,Setor,Part. feminina (%),Gap salarial (%),Salário médio F (R$)
13,Outros serviços,58.07,68.35,2381.06
14,Financeiro,58.18,64.79,6009.40
8,Artes e cultura,46.45,51.76,1888.27
18,Adm. pública,65.25,44.02,4821.75
6,Ind. transformação,35.60,43.41,2362.00
19,Educação,66.44,40.41,4048.98
10,Administrativo,50.11,39.41,1848.34
20,Saúde,81.20,38.96,3127.92
7,TIC,36.68,37.10,3830.09
3,Eletricidade e gás,19.74,32.70,6777.07


In [11]:
import pandas as pd
import plotly.express as px

# 1. Usar o arquivo de vagas (CAGED), que reflete melhor a realidade do mercado
df_vagas = pd.read_csv(config.gold_dir / "vagas_saldo_por_uf_perfil.csv", encoding="utf-8")

# 2. Filtrar apenas os sexos definidos (para evitar categorias 'Não Identificado')
df_vagas = df_vagas[df_vagas["sexo"].isin(["Masculino", "Feminino"])]

# 3. Em vez de contar linhas, vamos somar o estoque/saldo para ver a ocupação
df_part_real = (
    df_vagas.groupby(["sigla_uf", "sexo"])["saldo"]
    .sum()
    .abs() # Usamos o valor absoluto para representar volume de movimentação/presença
    .reset_index(name="volume")
)

# 4. Calcular o percentual real por estado
df_part_real["proporcao"] = df_part_real.groupby("sigla_uf")["volume"].transform(
    lambda x: (x / x.sum()) * 100
)

# 5. Gerar o gráfico com dados que não serão "50/50"
fig_part_real = px.bar(
    df_part_real,
    x="sigla_uf",
    y="proporcao",
    color="sexo",
    title="Participação Real no Mercado de Trabalho (CAGED) — Região Sul",
    labels={"proporcao": "Participação (%)", "sigla_uf": "Estado", "sexo": "Sexo"},
    text=df_part_real["proporcao"].apply(lambda x: f"{x:.1f}%"),
    color_discrete_map={"Masculino": "#4C72B0", "Feminino": "#DD8452"},
    barmode="relative"
)

fig_part_real.update_layout(yaxis_range=[0, 100], template="plotly_white")
fig_part_real.show()

print("Agora os dados devem refletir a assimetria real do mercado do Sul:")
display(df_part_real.pivot(index="sigla_uf", columns="sexo", values="proporcao").round(2))

Agora os dados devem refletir a assimetria real do mercado do Sul:


sexo,Feminino,Masculino
sigla_uf,,
PR,49.13,50.87
RS,52.94,47.06
SC,48.73,51.27


In [12]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Carregar o arquivo disponível na GOLD
df_base = pd.read_csv(config.gold_dir / "gap_salarial_por_uf_sexo_idade.csv", encoding="utf-8")

# 2. Reestruturar os dados para o formato longo (melt) para facilitar os cálculos
df_salarios = df_base.melt(
    id_vars=['uf_nome', 'periodo'],
    value_vars=['salario_feminino', 'salario_masculino'],
    var_name='sexo',
    value_name='salario'
)

df_salarios['sexo'] = df_salarios['sexo'].map({
    'salario_feminino': 'Feminino',
    'salario_masculino': 'Masculino'
})

# 3. Calcular a Média e a Mediana por Estado e por Sexo
df_stats = df_salarios.groupby(['uf_nome', 'sexo'])['salario'].agg(['mean', 'median']).reset_index()
df_stats.columns = ['Estado', 'Sexo', 'Média (R$)', 'Mediana (R$)']

# Arredondar os valores para 2 casas decimais
df_stats['Média (R$)'] = df_stats['Média (R$)'].round(2)
df_stats['Mediana (R$)'] = df_stats['Mediana (R$)'].round(2)

# Calcular a diferença/distorção causada pelos salários altos
df_stats['Distorção (Média - Mediana)'] = (df_stats['Média (R$)'] - df_stats['Mediana (R$)']).round(2)

# 4. Exibir a tabela com o resumo analítico
print("=== Análise Estatística: Média vs. Mediana Salarial ===")
display(df_stats)

# 5. Criar um gráfico comparativo interativo usando Plotly Graph Objects
fig_comp = go.Figure()

# Adicionar as barras de Média
for sexo, cor in [('Masculino', '#4C72B0'), ('Feminino', '#DD8452')]:
    df_sub = df_stats[df_stats['Sexo'] == sexo]
    
    # Barras para a Média
    fig_comp.add_trace(go.Bar(
        name=f'Média - {sexo}',
        x=df_sub['Estado'],
        y=df_sub['Média (R$)'],
        marker_color=cor,
        opacity=0.85,
        text=df_sub['Média (R$)'].apply(lambda x: f"R$ {x:,.2f}"),
        textposition='outside'
    ))
    
    # Linhas/Pontos para marcar a Mediana e mostrar a diferença
    fig_comp.add_trace(go.Scatter(
        name=f'Mediana - {sexo}',
        x=df_sub['Estado'],
        y=df_sub['Mediana (R$)'],
        mode='markers',
        marker=dict(color=cor, size=12, symbol='diamond', line=dict(width=2, color='black')),
        hovertemplate="Estado: %{x}<br>Mediana: R$ %{y:,.2f}"
    ))

# 6. Ajustes de layout para apresentação acadêmica
fig_comp.update_layout(
    title="Comparação entre Média e Mediana Salarial por Sexo — Região Sul",
    xaxis_title="Estado",
    yaxis_title="Rendimento Médio (R$)",
    barmode='group',
    template="plotly_white",
    legend_title_text="Métrica e Sexo",
    hovermode="x unified"
)

fig_comp.show()

=== Análise Estatística: Média vs. Mediana Salarial ===


,Estado,Sexo,Média (R$),Mediana (R$),Distorção (Média - Mediana)
0,Paraná,Feminino,1530.62,1521.25,9.37
1,Paraná,Masculino,2040.39,2023.71,16.68
2,Rio Grande do Sul,Feminino,1571.21,1570.86,0.35
3,Rio Grande do Sul,Masculino,2057.27,2045.15,12.12
4,Santa Catarina,Feminino,1627.08,1587.12,39.96
5,Santa Catarina,Masculino,2124.26,2076.54,47.72


In [14]:
import sys
import pathlib
import pandas as pd

# 1. Configurar caminhos do ambiente
_root = pathlib.Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import config

print("=== DIAGNÓSTICO DE ARQUIVOS DISPONÍVEIS ===")

# Listar arquivos na pasta GOLD
print("\n[Arquivos na pasta GOLD]:")
arquivos_gold = list(config.gold_dir.glob("*.csv"))
for f in arquivos_gold:
    try:
        df_temp = pd.read_csv(f, nrows=2, encoding="utf-8")
        print(f"-> Nome: {f.name}")
        print(f"   Colunas: {df_temp.columns.tolist()}\n")
    except Exception as e:
        print(f"-> Erro ao ler {f.name}: {e}")

# Listar arquivos na pasta SILVER
print("\n[Arquivos na pasta SILVER]:")
arquivos_silver = list(config.silver_dir.glob("*.csv"))
for f in arquivos_silver:
    try:
        df_temp = pd.read_csv(f, nrows=2, encoding="utf-8")
        print(f"-> Nome: {f.name}")
        print(f"   Colunas: {df_temp.columns.tolist()}\n")
    except Exception as e:
        print(f"-> Erro ao ler {f.name}: {e}")

=== DIAGNÓSTICO DE ARQUIVOS DISPONÍVEIS ===

[Arquivos na pasta GOLD]:
-> Nome: distribuicao_ocupacional.csv
   Colunas: ['ano', 'sigla_uf', 'cbo_2002', 'sexo', 'n_vinculos', 'salario_medio', 'total_vinculos', 'share_pct']

-> Nome: gap_salarial_por_escolaridade.csv
   Colunas: ['sigla_uf', 'ano', 'escolaridade', 'salario_feminino', 'salario_masculino', 'gap_pct', 'participacao_feminina_pct']

-> Nome: gap_salarial_por_setor.csv
   Colunas: ['sigla_uf', 'ano', 'setor', 'sexo', 'n_vinculos', 'salario_medio', 'total_vinculos', 'share_pct', 'gap_pct']

-> Nome: gap_salarial_por_uf_sexo_idade.csv
   Colunas: ['uf_cod', 'uf_nome', 'periodo', 'salario_feminino', 'salario_masculino', 'gap_pct']

-> Nome: vagas_saldo_por_uf_perfil.csv
   Colunas: ['ano', 'mes', 'sigla_uf', 'sexo', 'faixa_etaria', 'admissoes', 'desligamentos', 'saldo', 'salario_medio_admissao']


[Arquivos na pasta SILVER]:
-> Nome: caged_limpo.csv
   Colunas: ['ano', 'mes', 'sigla_uf', 'sexo', 'grau_instrucao', 'cbo_2002', 'cn

In [17]:
import sys
import pathlib
import pandas as pd
import plotly.graph_objects as go

# 1. Configurar caminhos do ambiente
_root = pathlib.Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import config

# 2. Carregar o arquivo bruto da pasta SILVER para extrair as medianas reais
print("Carregando dados da SILVER (isso pode levar alguns segundos devido ao volume da RAIS)...")
df_bruto = pd.read_csv(config.silver_dir / "rais_limpo.csv", encoding="utf-8")

# 3. Mapeamento das grandes seções do CNAE (Setores) baseado nos códigos subclasse
def mapear_cnae_para_setor(codigo):
    str_coda = str(codigo).zfill(5)
    prefixo = int(str_coda[:2])
    
    if prefixo <= 3: return "Agropecuária"
    elif prefixo <= 9: return "Ind. extrativa"
    elif prefixo <= 33: return "Ind. transformation"
    elif prefixo == 35: return "Eletricidade e gás"
    elif prefixo <= 39: return "Água e saneamento"
    elif prefixo <= 43: return "Construção"
    elif prefixo <= 47: return "Comércio"
    elif prefixo <= 53: return "Transporte"
    elif prefixo <= 56: return "Alojamento e alimentação"
    elif prefixo <= 63: return "TIC"
    elif prefixo <= 66: return "Financeiro"
    elif prefixo == 68: return "Imobiliário"
    elif prefixo <= 75: return "Prof. e técnico"
    elif prefixo <= 82: return "Administrativo"
    elif prefixo == 84: return "Adm. pública"
    elif prefixo == 85: return "Educação"
    elif prefixo <= 88: return "Saúde"
    elif prefixo <= 93: return "Artes e cultura"
    elif prefixo <= 96: return "Outros serviços"
    elif prefixo == 97: return "Serv. domésticos"
    else: return "Outros"

# Aplicar o mapeamento para criar a coluna de Setor legível (Corrigido o nome da função aqui)
df_bruto['Setor'] = df_bruto['cnae_2_subclasse'].apply(mapear_cnae_para_setor)

# 4. Calcular as Medianas (Feminina, Masculina e Geral) por Setor
print("Calculando as medianas salariais por gênero...")

# Mediana por Gênero
df_med_sexo = df_bruto.groupby(['Setor', 'sexo'])['valor_remuneracao_media'].median().reset_index()
df_pivot_med = df_med_sexo.pivot(index='Setor', columns='sexo', values='valor_remuneracao_media').reset_index()

# Garantir padronização das colunas do pivot
col_f = 'Feminino' if 'Feminino' in df_pivot_med.columns else ('feminino' if 'feminino' in df_pivot_med.columns else df_pivot_med.columns[1])
col_m = 'Masculino' if 'Masculino' in df_pivot_med.columns else ('masculino' if 'masculino' in df_pivot_med.columns else df_pivot_med.columns[2])

df_pivot_med = df_pivot_med.rename(columns={
    col_f: 'Mediana Feminina (R$)',
    col_m: 'Mediana Masculina (R$)'
})

# Mediana Geral (Independente de sexo)
df_med_geral = df_bruto.groupby('Setor')['valor_remuneracao_media'].median().reset_index(name='Mediana Geral (R$)')

# Unificar tabelas
df_setor_medianas = df_pivot_med.merge(df_med_geral, on='Setor')

# Ordenar para o gráfico ficar elegante
df_setor_medianas = df_setor_medianas.sort_values(by='Mediana Geral (R$)', ascending=True)

# 5. Exibir a tabela com as medianas reais no console
print("\n=== Tabela Analítica: Medianas Salariais por Setor ===")
display(df_setor_medianas.round(2))

# 6. Construir o gráfico interativo de barras horizontais
fig_medianas = go.Figure()

# Barra da Mediana Feminina
fig_medianas.add_trace(go.Bar(
    y=df_setor_medianas['Setor'],
    x=df_setor_medianas['Mediana Feminina (R$)'],
    name='Mediana Feminina',
    orientation='h',
    marker_color='#DD8452',
    text=df_setor_medianas['Mediana Feminina (R$)'].apply(lambda x: f"R$ {x:,.2f}" if pd.notnull(x) else ""),
    textposition='inside'
))

# Barra da Mediana Masculina
fig_medianas.add_trace(go.Bar(
    y=df_setor_medianas['Setor'],
    x=df_setor_medianas['Mediana Masculina (R$)'],
    name='Mediana Masculina',
    orientation='h',
    marker_color='#4C72B0',
    text=df_setor_medianas['Mediana Masculina (R$)'].apply(lambda x: f"R$ {x:,.2f}" if pd.notnull(x) else ""),
    textposition='inside'
))

# Marcador Losango para a Mediana Geral (Total Combinado)
fig_medianas.add_trace(go.Scatter(
    y=df_setor_medianas['Setor'],
    x=df_setor_medianas['Mediana Geral (R$)'],
    name='Mediana Geral (Total)',
    mode='markers',
    marker=dict(color='#2ca02c', size=10, symbol='diamond', line=dict(width=1, color='black')),
    hovertemplate="Setor: %{y}<br>Mediana Geral: R$ %{x:,.2f}"
))

# Ajustes estéticos finais
fig_medianas.update_layout(
    title="Distribuição das Medianas Salariais por Setor Econômico e Gênero — Região Sul",
    xaxis_title="Rendimento Mediano (R$)",
    yaxis_title="Setor Econômico",
    barmode='group',
    template="plotly_white",
    height=800,
    legend_title_text="Métrica",
    hovermode="y unified"
)

fig_medianas.show()

Carregando dados da SILVER (isso pode levar alguns segundos devido ao volume da RAIS)...
Calculando as medianas salariais por gênero...

=== Tabela Analítica: Medianas Salariais por Setor ===


,Setor,Mediana Feminina (R$),Mediana Masculina (R$),Mediana Geral (R$)
15,Serv. domésticos,1349.46,1484.08,1400.00
2,Alojamento e alimentação,1660.40,1806.67,1707.77
1,Administrativo,1536.93,2024.69,1741.52
3,Artes e cultura,1646.69,1997.93,1816.50
12,Outros serviços,1884.82,1830.07,1866.04
9,Imobiliário,1824.21,2025.60,1892.06
4,Comércio,1834.16,2064.58,1943.95
13,Prof. e técnico,2050.18,2306.10,2153.69
5,Construção,1932.24,2184.87,2160.00
10,Ind. transformation,1916.02,2409.31,2197.53


In [19]:
import sys
import pathlib
import pandas as pd
import plotly.graph_objects as go

# 1. Configurar caminhos do ambiente
_root = pathlib.Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import config

# 2. Carregar o arquivo correto da pasta GOLD
df_setores = pd.read_csv(config.gold_dir / "gap_salarial_por_setor.csv", encoding="utf-8")

# 3. Mapear setores para nomes mais amigáveis (caso estejam em códigos ou siglas)
# O pandas vai agrupar os salários por Setor e Sexo
df_Pivot = df_setores.pivot_table(
    index='setor', 
    columns='sexo', 
    values='salario_medio', 
    aggfunc='mean'
).reset_index()

# Calcular a Média Geral (combinada) ponderada ou simples por setor
df_geral = df_setores.groupby('setor')['salario_medio'].mean().reset_index(name='Média Geral')

# Unificar os dados em uma tabela limpa para conferência
df_setor_stats = df_Pivot.merge(df_geral, on='setor')

# Ajustar nomes das colunas
# Dependendo de como os dados estão descritos na coluna 'sexo', o pivot criará colunas específicas
# Vamos garantir que as colunas existam tratando possíveis variações de maiúscula/minúscula
col_f = 'Feminino' if 'Feminino' in df_setor_stats.columns else ('feminino' if 'feminino' in df_setor_stats.columns else df_setor_stats.columns[1])
col_m = 'Masculino' if 'Masculino' in df_setor_stats.columns else ('masculino' if 'masculino' in df_setor_stats.columns else df_setor_stats.columns[2])

df_setor_stats = df_setor_stats.rename(columns={
    'setor': 'Setor',
    col_f: 'Média Feminina (R$)',
    col_m: 'Média Masculina (R$)'
})

# Ordenar pelos setores com maior média geral para um visual mais organizado
df_setor_stats = df_setor_stats.sort_values(by='Média Geral', ascending=True)

# 4. Exibir a tabela no console do Jupyter
print("=== Rendimentos Médios por Setor Econômico — Região Sul ===")
display(df_setor_stats.round(2))

# 5. Criar o Gráfico de Barras Horizontais Comparativo
fig_setores = go.Figure()

# Barra da Remuneração Feminina
fig_setores.add_trace(go.Bar(
    y=df_setor_stats['Setor'],
    x=df_setor_stats['Média Feminina (R$)'],
    name='Média Feminina',
    orientation='h',
    marker_color='#DD8452',
    text=df_setor_stats['Média Feminina (R$)'].apply(lambda x: f"R$ {x:,.2f}" if pd.notnull(x) else ""),
    textposition='inside'
))

# Barra da Remuneração Masculina
fig_setores.add_trace(go.Bar(
    y=df_setor_stats['Setor'],
    x=df_setor_stats['Média Masculina (R$)'],
    name='Média Masculina',
    orientation='h',
    marker_color='#4C72B0',
    text=df_setor_stats['Média Masculina (R$)'].apply(lambda x: f"R$ {x:,.2f}" if pd.notnull(x) else ""),
    textposition='inside'
))

# Linha de Marcadores para a Média Geral (Total)
fig_setores.add_trace(go.Scatter(
    y=df_setor_stats['Setor'],
    x=df_setor_stats['Média Geral'],
    name='Média Geral (Total)',
    mode='markers',
    marker=dict(color='#2ca02c', size=10, symbol='diamond', line=dict(width=1, color='black')),
    hovertemplate="Setor: %{y}<br>Média Geral: R$ %{x:,.2f}"
))

# 6. Ajustes de Layout para o padrão acadêmico do Trabalho Integrador
fig_setores.update_layout(
    title="Rendimento Médio por Setor Econômico e Gênero — Região Sul",
    xaxis_title="Salário Médio (R$)",
    yaxis_title="Setor Econômico",
    barmode='group',
    template="plotly_white",
    height=800, 
    legend_title_text="Categoria",
    hovermode="y unified"
)

fig_setores.show()

=== Rendimentos Médios por Setor Econômico — Região Sul ===


,Setor,Média Feminina (R$),Média Masculina (R$),Média Geral
17,Serv. domésticos,1261.83,1339.83,1300.83
3,Alojamento e alimentação,1726.52,1973.33,1849.92
2,Agropecuária,1946.05,2394.35,2170.20
1,Administrativo,1848.34,2571.41,2209.87
4,Artes e cultura,1888.27,2849.97,2369.12
5,Comércio,2188.85,2731.78,2460.32
10,Imobiliário,2270.34,2816.15,2543.25
6,Construção,2571.54,2657.41,2614.47
19,Transporte,2480.46,2869.24,2674.85
12,Ind. transformação,2362.00,3389.98,2875.99


In [23]:
import sys
import pathlib
import pandas as pd
import plotly.express as px

# 1. Configurar caminhos do ambiente
_root = pathlib.Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import config

print("Carregando base da SILVER para processamento das barras...")
df_bruto = pd.read_csv(config.silver_dir / "rais_limpo.csv", encoding="utf-8")

# 2. Mapeamento dos códigos CBO para Grandes Grupos Ocupacionais
def mapear_cbo_para_grupo(codigo):
    try:
        digito = int(str(codigo)[0])
        mapeamento = {
            0: "Militares/Policiais",
            1: "Diretores/Gerentes",
            2: "Prof. Ciências/Artes (Seniors)",
            3: "Técnicos Nível Médio",
            4: "Trabalhadores Adm.",
            5: "Serviços/Comércio",
            6: "Agropecuária/Pesca",
            7: "Industriais (Produção)",
            8: "Industriais (Operação)",
            9: "Reparação/Manutenção"
        }
        return mapeamento.get(digito, "Outros")
    except:
        return "Não Identificado"

df_bruto['Função'] = df_bruto['cbo_2002'].apply(mapear_cbo_para_grupo)

# Adicionar nome amigável para os estados
_UF_MAP = {41: "Paraná", 42: "Santa Catarina", 43: "Rio Grande do Sul"}
if 'sigla_uf' in df_bruto.columns:
    df_bruto['Estado'] = df_bruto['sigla_uf'].map({'PR': 'Paraná', 'SC': 'Santa Catarina', 'RS': 'Rio Grande do Sul'})
else:
    df_bruto['Estado'] = df_bruto['uf_cod'].map(_UF_MAP)

# 3. Calcular a Participação Feminina (%)
df_counts = df_bruto.groupby(['Estado', 'Função', 'sexo']).size().reset_index(name='vinc_sexo')
df_total_grupo = df_bruto.groupby(['Estado', 'Função']).size().reset_index(name='total_grupo')
df_part = df_counts.merge(df_total_grupo, on=['Estado', 'Função'])
df_part['Participação Feminina (%)'] = (df_part['vinc_sexo'] / df_part['total_grupo']) * 100
df_part_f = df_part[df_part['sexo'].str.lower().str.startswith('f')].copy()

# 4. Calcular as Medianas (Geral e Feminina)
df_med_geral = df_bruto.groupby(['Estado', 'Função'])['valor_remuneracao_media'].median().reset_index(name='Mediana Geral (R$)')
df_med_fem = df_bruto[df_bruto['sexo'].str.lower().str.startswith('f')].groupby(['Estado', 'Função'])['valor_remuneracao_media'].median().reset_index(name='Mediana Feminina (R$)')

# 5. Unificar os dados
df_cruzado = df_part_f[['Estado', 'Função', 'Participação Feminina (%)']].merge(df_med_geral, on=['Estado', 'Função'])
df_cruzado = df_cruzado.merge(df_med_fem, on=['Estado', 'Função'])

# Formatar os textos das etiquetas que vão dentro das barras
df_cruzado['Dados'] = df_cruzado.apply(
    lambda r: f"Part. F: {r['Participação Feminina (%)']:.1f}% | Med. F: R${r['Mediana Feminina (R$)']:.0f}", axis=1
)

# Ordenar para que as funções fiquem organizadas da maior para a menor mediana geral
df_cruzado = df_cruzado.sort_values(by='Mediana Geral (R$)', ascending=True)

# 6. Criar o Gráfico de Barras Horizontais Facetado (Separado por Estado)
fig_barras_complexas = px.bar(
    df_cruzado,
    x="Mediana Geral (R$)",
    y="Função",
    color="Mediana Feminina (R$)", # Gradiente de cor indica o ganho real das mulheres
    facet_col="Estado", # Cria 3 colunas de gráficos (PR, SC, RS) lado a lado
    title="Análise Comparativa Ocupacional: Mediana Geral, Mediana Feminina e Participação por Estado",
    labels={
        "Mediana Geral (R$)": "Mediana Salarial Geral (R$)",
        "Função": "Grupo Ocupacional",
        "Mediana Feminina (R$)": "Mediana Feminina"
    },
    text="Dados", # Exibe a porcentagem e o ganho das mulheres escrito na barra
    color_continuous_scale="Viridis", # Escala de cor elegante e legível
    template="plotly_white"
)

# Ajustes finos de layout para caber os textos dentro das barras horizontais
fig_barras_complexas.update_traces(textposition='inside', insidetextanchor='start')
fig_barras_complexas.update_layout(
    height=750,
    coloraxis_colorbar_title="Mediana Feminina (R$)",
    margin=dict(l=50, r=50, t=80, b=50)
)

# Remover os prefixos automáticos "Estado=" do topo de cada coluna do gráfico
fig_barras_complexas.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig_barras_complexas.show()

Carregando base da SILVER para processamento das barras...


In [26]:
# ==========================================
# 6. Criar o Gráfico de Barras Horizontais Facetado (Separado por Estado)
# ==========================================
fig_barras_complexas = px.bar(
    df_cruzado,
    x="Mediana Geral (R$)",
    y="Função",
    color="Mediana Feminina (R$)", 
    facet_col="Estado", 
    title="Análise Comparativa Ocupacional: Mediana Geral, Mediana Feminina e Participação por Estado",
    labels={
        "Mediana Geral (R$)": "Mediana Salarial Geral (R$)",
        "Função": "Grupo Ocupacional",
        "Mediana Feminina (R$)": "Mediana Feminina"
    },
    text="Dados", 
    color_continuous_scale="Viridis", 
    template="plotly_white"
)

fig_barras_complexas.update_traces(textposition='inside', insidetextanchor='start')
fig_barras_complexas.update_layout(
    height=750,
    coloraxis_colorbar_title="Mediana Feminina (R$)",
    margin=dict(l=50, r=50, t=80, b=50)
)

fig_barras_complexas.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_barras_complexas.show()


# ==========================================
# 7. GERAR A TABELA ACADÊMICA CONSOLIDADA
# ==========================================
# Criamos uma cópia limpa para formatar os dados de texto do relatório
df_tabela = df_cruzado.copy()

# Renomear colunas para o padrão de exibição do TCC
df_tabela = df_tabela.rename(columns={
    'Participação Feminina (%)': 'Part. Feminina (%)',
    'Mediana Geral (R$)': 'Med. Geral (R$)',
    'Mediana Feminina (R$)': 'Med. Feminina (R$)'
})

# Ordenar de forma hierárquica por Estado e depois pelas maiores funções
df_tabela = df_tabela.sort_values(by=['Estado', 'Med. Geral (R$)'], ascending=[True, False])

print("\n" + "="*80)
print("  TABELA RESUMO MULTIVARIADA: MERCADO DE TRABALHO DA REGIÃO SUL")
print("="*80)

# Exibir a tabela no console formatando os valores numéricos com duas casas decimais
pd.set_option('display.max_rows', 50)
display(df_tabela[['Estado', 'Função', 'Part. Feminina (%)', 'Med. Geral (R$)', 'Med. Feminina (R$)']].reset_index(drop=True))


  TABELA RESUMO MULTIVARIADA: MERCADO DE TRABALHO DA REGIÃO SUL


,Estado,Função,Part. Feminina (%),Med. Geral (R$),Med. Feminina (R$)
0,Paraná,Prof. Ciências/Artes (Seniors),64.390862,4382.090,4034.370
1,Paraná,Diretores/Gerentes,43.276290,4134.200,3636.845
2,Paraná,Técnicos Nível Médio,58.635751,2898.500,2735.850
3,Paraná,Reparação/Manutenção,8.262078,2311.170,1676.110
4,Paraná,Industriais (Operação),31.046591,2121.330,1888.870
5,Paraná,Industriais (Produção),17.013048,2095.220,1734.200
6,Paraná,Trabalhadores Adm.,60.668427,1927.730,1861.310
7,Paraná,Agropecuária/Pesca,15.057854,1891.330,1661.410
8,Paraná,Serviços/Comércio,56.737731,1775.730,1655.690
9,Paraná,Não Identificado,62.573099,1503.630,1636.060


In [24]:
import sys
import pathlib
import pandas as pd

# 1. Configurar caminhos do ambiente
_root = pathlib.Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import config

print("Gerando a tabela analítica de suporte...")

# O DataFrame 'df_cruzado' já possui os dados calculados do passo anterior.
# Caso você esteja rodando essa célula separadamente, garantimos a ordenação:
df_tabela = df_cruzado.sort_values(by=['Estado', 'Mediana Geral (R$)'], ascending=[True, False]).copy()

# 2. Renomear e selecionar as colunas para o formato acadêmico
df_tabela_formatada = df_tabela[[
    'Estado', 
    'Função', 
    'Participação Feminina (%)', 
    'Mediana Feminina (R$)', 
    'Mediana Geral (R$)'
]].copy()

# 3. Calcular a Mediana Masculina para deixar a tabela ainda mais completa
df_med_masc = df_bruto[df_bruto['sexo'].str.lower().str.startswith('m')].groupby(['Estado', 'Função'])['valor_remuneracao_media'].median().reset_index(name='Mediana Masculina (R$)')
df_med_masc['Mediana Masculina (R$)'] = df_med_masc['Mediana Masculina (R$)'].round(2)

# Unificar a mediana masculina na tabela
df_tabela_formatada = df_tabela_formatada.merge(df_med_masc, on=['Estado', 'Função'], how='left')

# Reorganizar a ordem das colunas para leitura lógica
df_tabela_formatada = df_tabela_formatada[[
    'Estado', 
    'Função', 
    'Participação Feminina (%)', 
    'Mediana Feminina (R$)', 
    'Mediana Masculina (R$)', 
    'Mediana Geral (R$)'
]]

# 4. Configurar a exibição para o Jupyter não cortar as linhas
pd.set_option('display.max_rows', 100)

print("\n=== TABELA SUPORTE: DADOS OCUPACIONAIS POR ESTADO ===")
display(df_tabela_formatada)

# 5. Opcional: Salvar uma cópia em CSV na sua pasta GOLD para segurança ou uso no Word/Excel
# df_tabela_formatada.to_csv(config.gold_dir / "tabela_suporte_grafico_complexo.csv", index=False, encoding="utf-8")

Gerando a tabela analítica de suporte...

=== TABELA SUPORTE: DADOS OCUPACIONAIS POR ESTADO ===


,Estado,Função,Participação Feminina (%),Mediana Feminina (R$),Mediana Masculina (R$),Mediana Geral (R$)
0,Paraná,Prof. Ciências/Artes (Seniors),64.390862,4034.370,5376.37,4382.090
1,Paraná,Diretores/Gerentes,43.276290,3636.845,4635.85,4134.200
2,Paraná,Técnicos Nível Médio,58.635751,2735.850,3203.74,2898.500
3,Paraná,Reparação/Manutenção,8.262078,1676.110,2411.72,2311.170
4,Paraná,Industriais (Operação),31.046591,1888.870,2274.49,2121.330
5,Paraná,Industriais (Produção),17.013048,1734.200,2223.79,2095.220
6,Paraná,Trabalhadores Adm.,60.668427,1861.310,2056.25,1927.730
7,Paraná,Agropecuária/Pesca,15.057854,1661.410,1945.12,1891.330
8,Paraná,Serviços/Comércio,56.737731,1655.690,1982.01,1775.730
9,Paraná,Não Identificado,62.573099,1636.060,1373.22,1503.630
